<a href="https://colab.research.google.com/github/IT24102850/srilanka-flood-warning/blob/main/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **01_data_collection**

Install required libraries — paste this in cell 1

In [1]:
!pip install pandas numpy geopandas folium xgboost
!pip install matplotlib seaborn scikit-learn shap
!pip install requests tqdm

Colab code to download NASA GPM IMERG rainfall data for Sri Lanka bounding box 2000 to 2023


In [2]:
!pip install earthaccess xarray netCDF4 h5py rasterio geopandas folium -q
import earthaccess
import xarray as xr
import numpy as np
import pandas as pd
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 82.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.


In [3]:
earthaccess.login(strategy="interactive")

Enter your Earthdata Login username: HASIRU
Enter your Earthdata password: ··········


Search for GPM IMERG monthly data over Sri Lanka

In [6]:
# Sri Lanka bounding box: west, south, east, north

BBOX = (79.5, 5.9, 81.9, 9.8)

# Try version 07 (NASA retired v06 in 2024)
results = earthaccess.search_data(
    short_name="GPM_3IMERGM",
    version="07",
    temporal=("2000-06-01", "2023-12-31"),
    bounding_box=BBOX
)

print(f"Found {len(results)} monthly files")

Found 283 monthly files


/usr/local/lib/python3.12/dist-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Download (subset to Sri Lanka only to save space)

In [8]:
# Re-login to refresh credentials
earthaccess.login(strategy="interactive")

In [9]:
# Download — should work now
os.makedirs("data/raw/gpm", exist_ok=True)
files = earthaccess.download(results, "data/raw/gpm")
print(f"Downloaded {len(files)} files")

QUEUEING TASKS | :   0%|          | 0/283 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/283 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/283 [00:00<?, ?it/s]

Downloaded 283 files


In [10]:
import glob

records = []

for fpath in sorted(glob.glob("data/raw/gpm/*.HDF5")):
    try:
        ds = xr.open_dataset(fpath, group="Grid", engine="netcdf4")
        rain = ds['precipitation'].sel(
            lon=slice(79.5, 81.9),
            lat=slice(5.9, 9.8)
        )
        hours_in_month = int(ds.time.dt.days_in_month.values[0]) * 24
        monthly_mean_mm = float(rain.mean().values) * hours_in_month
        year  = int(ds.time.dt.year.values[0])
        month = int(ds.time.dt.month.values[0])
        records.append({"year": year, "month": month, "rainfall_mm": round(monthly_mean_mm, 2)})
        ds.close()
    except Exception as e:
        print(f"Skipped {fpath}: {e}")

rainfall_df = pd.DataFrame(records).sort_values(["year","month"]).reset_index(drop=True)
print(f"Shape: {rainfall_df.shape}")
print(rainfall_df.head(12))

Shape: (283, 3)
    year  month  rainfall_mm
0   2000      6        79.90
1   2000      7        17.47
2   2000      8        96.13
3   2000      9       144.87
4   2000     10       114.11
5   2000     11       256.55
6   2000     12       217.79
7   2001      1       167.20
8   2001      2        66.32
9   2001      3        12.48
10  2001      4       204.32
11  2001      5        58.24


save and verify

In [11]:
os.makedirs("data/processed", exist_ok=True)
rainfall_df.to_csv("data/processed/gpm_monthly_srilanka.csv", index=False)

print("Average rainfall by month (mm):")
print(rainfall_df.groupby("month")["rainfall_mm"].mean().round(1))

Average rainfall by month (mm):
month
1     100.3
2      73.4
3      82.4
4     139.3
5     127.9
6      61.9
7      54.1
8      71.1
9     111.0
10    248.6
11    308.4
12    237.1
Name: rainfall_mm, dtype: float64
